In [1]:
# NOTEBOOK 02b: INGREDIENT QUALITY CHECK

import os
import re
from collections import Counter

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

PROCESSED_PATH = '../data/processed'
QUALITY_PATH   = '../data/quality_check'
os.makedirs(QUALITY_PATH, exist_ok=True)


In [2]:
# ----------------------------------------------------------
# STEP 0: LOAD CLEANED PRODUCT DATA
# ----------------------------------------------------------

products = pd.read_csv(
    os.path.join(PROCESSED_PATH, 'products_cleaned.csv')
)

products['ingredients_list'] = products['ingredients_parsed'].apply(
    lambda x: [
        i.strip()
        for i in str(x).split('|')
        if i.strip() != '' and i.strip().lower() != 'nan'
    ]
    if pd.notna(x) else []
)

all_ingredients    = [
    ing
    for sublist in products['ingredients_list']
    for ing in sublist
]
freq               = Counter(all_ingredients)
unique_ingredients = sorted(set(all_ingredients))

print("STEP 0: Load Cleaned Product Data")
print()
print(f"  {'Metric':<40} {'Value':>12}")
print(f"  {'----------':<40} {'-----':>12}")
print(f"  {'Products in dataset':<40} {len(products):>12,}")
print(f"  {'Total ingredient mentions':<40} {len(all_ingredients):>12,}")
print(f"  {'Total unique ingredients':<40} {len(unique_ingredients):>12,}")
print(f"  {'Average ingredients per product':<40} "
      f"{len(all_ingredients)/len(products):>12.1f}")

# Consistency check: compare upstream ingredient_count vs rebuilt list length.
# A mismatch means the parsing logic in this notebook diverged from upstream.
expected_total = products['ingredient_count'].sum()
rebuilt_total  = sum(len(lst) for lst in products['ingredients_list'])
match_status   = "MATCH" if expected_total == rebuilt_total else "MISMATCH"

print()
print("  Consistency Check (upstream vs rebuilt)")
print()
print(f"  {'Total from upstream ingredient_count':<40} {expected_total:>12,}")
print(f"  {'Total rebuilt from ingredients_list':<40} {rebuilt_total:>12,}")
print(f"  {'Status':<40} {match_status:>12}")


STEP 0: Load Cleaned Product Data

  Metric                                          Value
  ----------                                      -----
  Products in dataset                             7,544
  Total ingredient mentions                     261,561
  Total unique ingredients                       13,899
  Average ingredients per product                  34.7

  Consistency Check (upstream vs rebuilt)

  Total from upstream ingredient_count          261,559
  Total rebuilt from ingredients_list           261,561
  Status                                       MISMATCH


In [3]:
# ----------------------------------------------------------
# STEP 1: INGREDIENT COUNT CHECK
# ----------------------------------------------------------

print()
print("STEP 1: Ingredient Count Check")
print()

products['ingredient_count_check'] = (
    products['ingredients_list'].apply(len)
)

# Compare the reconstructed count with the upstream preprocessing count.
mismatch_mask = (
    products['ingredient_count_check'] != products['ingredient_count']
)

mismatch_rows = products.loc[
    mismatch_mask,
    [
        'product_id',
        'product_name',
        'brand_name',
        'ingredient_count',
        'ingredient_count_check',
        'ingredients_parsed'
    ]
]

upstream_mentions = int(products['ingredient_count'].sum())
reconstructed_mentions = len(all_ingredients)
mention_difference = reconstructed_mentions - upstream_mentions
mismatch_count = int(mismatch_mask.sum())

if mismatch_count == 0 and mention_difference == 0:
    count_check_status = "Counts fully aligned"
elif mismatch_count == 1 and mention_difference == 2:
    count_check_status = "Known two-entry reconstruction difference"
else:
    count_check_status = "Review required"

print(f"  Status                           : {count_check_status}")
print(f"  Products with count differences : {mismatch_count:,}")
print(f"  Upstream ingredient mentions    : {upstream_mentions:,}")
print(f"  Reconstructed ingredient mentions: {reconstructed_mentions:,}")
print(f"  Difference                      : {mention_difference:+,}")
print()
for _, row in mismatch_rows.iterrows():
    diff = row['ingredient_count_check'] - row['ingredient_count']
    print(f"Product : {row['product_name']}")
    print(f"Brand   : {row['brand_name']}")
    print(f"Upstream count : {row['ingredient_count']}")
    print(f"Rebuilt count  : {row['ingredient_count_check']}")
    print(f"Difference     : {diff:+d}")
    print(f"Parsed string  : {row['ingredients_parsed'][:300]}")
    print()

print()
print(f"  {'Statistic':<25} {'Value':>12}")
print(f"  {'----------':<25} {'-----':>12}")
print(f"  {'Minimum':<25} "
      f"{products['ingredient_count_check'].min():>12}")
print(f"  {'25th percentile':<25} "
      f"{products['ingredient_count_check'].quantile(0.25):>12.0f}")
print(f"  {'Median':<25} "
      f"{products['ingredient_count_check'].median():>12.0f}")
print(f"  {'75th percentile':<25} "
      f"{products['ingredient_count_check'].quantile(0.75):>12.0f}")
print(f"  {'Maximum':<25} "
      f"{products['ingredient_count_check'].max():>12}")
print(f"  {'Mean':<25} "
      f"{products['ingredient_count_check'].mean():>12.1f}")
print(f"  {'Std Dev':<25} "
      f"{products['ingredient_count_check'].std():>12.1f}")

low_count_products = products[
    products['ingredient_count_check'] <= 2
][[
    'product_id', 'product_name', 'brand_name',
    'ingredients', 'ingredients_parsed',
    'ingredient_count_check'
]]

print()
print(f"  {'Products with 0 ingredients':<40} "
      f"{(products['ingredient_count_check'] == 0).sum():>8,}")
print(f"  {'Products with 1-2 ingredients':<40} "
      f"{len(low_count_products):>8,}")

if len(low_count_products) > 0:
    print()
    print("  Sample products with very low ingredient count:")
    print()
    for _, row in low_count_products.head(3).iterrows():
        print(f"  Product : {row['product_name']}")
        print(f"  Brand   : {row['brand_name']}")
        print(f"  Raw     : {str(row['ingredients'])[:100]}")
        print(f"  Parsed  : {row['ingredients_parsed'][:80]}")
        print(f"  Count   : {row['ingredient_count_check']}")
        print()

low_count_products.to_csv(
    os.path.join(QUALITY_PATH,
                 'low_ingredient_count_products.csv'),
    index=False
)



STEP 1: Ingredient Count Check

  Status                           : Known two-entry reconstruction difference
  Products with count differences : 1
  Upstream ingredient mentions    : 261,559
  Reconstructed ingredient mentions: 261,561
  Difference                      : +2

Product : Transfer-proof Matte Liquid lipstick
Brand   : Gucci
Upstream count : 37
Rebuilt count  : 39
Difference     : +2
Parsed string  : isododecane | dimethicone | dimethicone/vinyl dimethicone crosspolymer | synthetic wax | cera microcristallina/microcrystalline wax/cire microcrystalline | mica | dicalcium phosphate | polyurethane-1 | tocopheryl acetate | caprylic/capric triglyceride | hexyldecanol | disteardimonium hectorite | di-


  Statistic                        Value
  ----------                       -----
  Minimum                              1
  25th percentile                     18
  Median                              28
  75th percentile                     40
  Maximum                       

In [4]:
# ----------------------------------------------------------
# STEP 2: PARSING ARTIFACT CHECK
# ----------------------------------------------------------

print()
print("STEP 2: Parsing Artifact Check")
print()

artifacts = []

for ing in unique_ingredients:
    issues = []

    if len(ing) <= 2:
        issues.append('too short')
    if re.match(r'^\d+\.?\d*$', ing):
        issues.append('purely numeric')
    if re.match(r'^\d+\.?\d*\s*%$', ing):
        issues.append('percentage value')
    if ing.startswith(')') or ing.startswith(']'):
        issues.append('starts with bracket')
    if ing.endswith('(') or ing.endswith('['):
        issues.append('ends with bracket')
    if ing.endswith('.'):
        issues.append('ends with period')
    if re.match(r'^[^a-zA-Z0-9]+$', ing):
        issues.append('no alphanumeric characters')
    if len(ing) > 80:
        issues.append('unusually long')

    if issues:
        artifacts.append({
            'ingredient': ing,
            'issue':      ', '.join(issues),
            'frequency':  freq[ing],
            'length':     len(ing)
        })

artifacts_df = pd.DataFrame(artifacts)

if len(artifacts_df) > 0:
    artifacts_df = artifacts_df.sort_values(
        ['frequency', 'length'], ascending=[False, False]
    )

artifact_rate = len(artifacts_df) / len(unique_ingredients) * 100

print(f"  {'Potential artifacts found':<40} {len(artifacts_df):>8,}")
print(f"  {'Percentage of unique ingredients':<40} "
      f"{artifact_rate:>7.1f}%")

if len(artifacts_df) > 0:
    print()
    print(f"  {'Ingredient':<50} {'Issue':<30} {'Freq':>8}")
    print(f"  {'----------':<50} {'-----':<30} {'----':>8}")
    for _, row in artifacts_df.head(20).iterrows():
        print(f"  {row['ingredient'][:48]:<50} "
              f"{row['issue']:<30} "
              f"{row['frequency']:>8,}")

artifacts_df.to_csv(
    os.path.join(QUALITY_PATH,
                 'potential_parsing_artifacts.csv'),
    index=False
)



STEP 2: Parsing Artifact Check

  Potential artifacts found                     168
  Percentage of unique ingredients             1.2%

  Ingredient                                         Issue                              Freq
  ----------                                         -----                              ----
  saccharomyces/camellia sinensis leaf/cladosiphon   unusually long                       27
  eucheuma serra/grateloupia sparsa/saccharina ang   unusually long                       11
  vegetable wax blend containing hydrogenated soyb   unusually long                        8
  rosmarinus officinalis (rosemary) leaf extract (   unusually long                        6
  euphorbia cerifera (candelilla) wax/candelilla c   unusually long                        5
  glycidoxypropyl trimethoxysilane/ methacryloyl p   unusually long                        3
  helianthus annuus (sunflower) seed oil (and) ros   unusually long                        3
  saccharum officinarum (

In [5]:
# ----------------------------------------------------------
# STEP 3: NON-INGREDIENT PHRASE CHECK
# ----------------------------------------------------------

print()
print("STEP 3: Non-Ingredient Phrase Check")
print()

phrase_patterns = [
    'free from', '-free', 'fragrance-free', 'without',
    'does not contain', 'no parabens', 'no phthalates',
    'may contain', 'made from', 'for external use',
    'avoid contact', 'cruelty-free', 'vegan',
    'dermatologist tested', 'hypoallergenic',
    'non-comedogenic', 'spf 15', 'spf 30', 'spf 50',
    'broad spectrum', 'uva/uvb', 'water resistant',
    'warning', 'keep out of reach', 'for use only',
    'active ingredient', 'inactive ingredient',
    'other ingredient', 'directions', 'dist. by',
    'manufactured by', 'mfg by', 'patent'
]

phrase_flags = []

for ing in unique_ingredients:
    matched = [p for p in phrase_patterns if p in ing.lower()]
    if matched:
        phrase_flags.append({
            'ingredient':      ing,
            'matched_pattern': ', '.join(matched),
            'frequency':       freq[ing],
            'length':          len(ing)
        })

phrase_flags_df = pd.DataFrame(phrase_flags)

if len(phrase_flags_df) > 0:
    phrase_flags_df = phrase_flags_df.sort_values(
        'frequency', ascending=False
    )

print(f"  {'Non-ingredient phrase candidates':<40} "
      f"{len(phrase_flags_df):>8,}")

if len(phrase_flags_df) > 0:
    print()
    print(f"  {'Ingredient':<50} {'Pattern':<30} {'Freq':>8}")
    print(f"  {'----------':<50} {'-------':<30} {'----':>8}")
    for _, row in phrase_flags_df.head(20).iterrows():
        print(f"  {row['ingredient'][:48]:<50} "
              f"{row['matched_pattern'][:28]:<30} "
              f"{row['frequency']:>8,}")

phrase_flags_df.to_csv(
    os.path.join(QUALITY_PATH,
                 'non_ingredient_phrase_candidates.csv'),
    index=False
)



STEP 3: Non-Ingredient Phrase Check

  Non-ingredient phrase candidates              100

  Ingredient                                         Pattern                            Freq
  ----------                                         -------                            ----
  all-natural coconut and beeswax blend. paraffin-   -free                                15
  and cruelty-free                                   -free, cruelty-free                  13
  may contain (+/-) red 40 lake/ci 16035             may contain                           8
  collagen (vegan)*                                  vegan                                 6
  potassium sorbate. may contain (peut contenir      may contain                           5
  phthalate-free fragrance                           -free                                 5
  collagen amino acids (vegan)*                      vegan                                 5
  paraffin-free 100% vegetable-wax blend containin   -free              

In [6]:
# ----------------------------------------------------------
# STEP 4: SYNONYM AND VARIANT CHECK
# ----------------------------------------------------------
# Step 4 is exploratory only, confirms synonym normalisation is needed.
# Full canonical mapping is built in Notebook 02c.

print()
print("STEP 4: Synonym and Variant Check")
print()

variant_groups = {
    'water':           ['water', 'aqua', 'eau'],
    'alcohol':         ['alcohol denat', 'denatured alcohol',
                        'sd alcohol', 'ethanol',
                        'isopropyl alcohol', 'benzyl alcohol'],
    'fragrance':       ['fragrance', 'parfum', 'perfume'],
    'vitamin e':       ['tocopherol', 'tocopheryl acetate',
                        'vitamin e'],
    'hyaluronic acid': ['hyaluronic acid', 'sodium hyaluronate'],
    'shea butter':     ['shea butter', 'butyrospermum parkii'],
    'titanium dioxide':['titanium dioxide', 'ci 77891'],
    'iron oxides':     ['iron oxides', 'ci 77491',
                        'ci 77492', 'ci 77499']
}

variant_records = []

print(f"  {'Ingredient Group':<25} {'Variants':>10} {'Mentions':>12}")
print(f"  {'----------------':<25} {'--------':>10} {'--------':>12}")

for group_name, terms in variant_groups.items():
    matches = []
    for ing in unique_ingredients:
        for term in terms:
            if term in ing.lower():
                matches.append(ing)
                break

    matches        = sorted(set(matches))
    total_mentions = sum(freq[m] for m in matches)

    print(f"  {group_name:<25} {len(matches):>10,} "
          f"{total_mentions:>12,}")
    print(f"    Examples: {matches[:4]}")

    for m in matches:
        variant_records.append({
            'variant_group':      group_name,
            'ingredient_variant': m,
            'frequency':          freq[m]
        })

variant_df = pd.DataFrame(variant_records)
variant_df.to_csv(
    os.path.join(QUALITY_PATH, 'ingredient_variant_groups.csv'),
    index=False
)

print()



STEP 4: Synonym and Variant Check

  Ingredient Group            Variants     Mentions
  ----------------            --------     --------
  water                            240        7,674
    Examples: ['(water', '* water/eau/aqua', '\\raqua (water)', 'adwoa blue tansy reparative mask water (aqua)']
  alcohol                          139        7,867
    Examples: ['(sd alcohol 40-b)', '*alcohol denat', '75% vol. alcohol denat', '78% vol. alcohol denat']
  fragrance                        112        4,360
    Examples: ['(fragrance)', '*fragrance (parfum)', '2 hexanediol parfum (fragrance)', '\\rparfum (fragrance)']
  vitamin e                         88        4,932
    Examples: ['***tocopherol', '*tocopherol (mixed)', 'and tocopherol', 'aqua / water glycerin dimethicone squalane bis-peg-18 methyl ether dimethyl silane sucrose stearate stearyl alcohol peg-8 stearate myristyl myristate prunus armeniaca kernel oil / apricot kernel oil phenoxyethanol persea gratissima oil / avocado 

In [7]:
# ----------------------------------------------------------
# STEP 5: KEY RISK INGREDIENT DETECTION
# ----------------------------------------------------------

print()
print("STEP 5: Key Risk Ingredient Detection")
print()

# Each entry: (search term, match_mode)
#   'sub'  = substring match (default; for clear multi-word INCI names)
#   'word' = whole-word match (for short tokens that would cause false
#            positives if matched as substrings, e.g. 'sls', 'sles', 'dehp')
key_ingredients = {
    'parabens': [
        ('paraben', 'sub'), ('methylparaben', 'sub'),
        ('propylparaben', 'sub'), ('butylparaben', 'sub'),
        ('ethylparaben', 'sub')
    ],
    'phthalates': [
        ('phthalate', 'sub'), ('dibutyl phthalate', 'sub'),
        ('diethyl phthalate', 'sub'), ('dehp', 'word')
    ],
    'formaldehyde releasers': [
        ('formaldehyde', 'sub'), ('dmdm hydantoin', 'sub'),
        ('imidazolidinyl urea', 'sub'),
        ('diazolidinyl urea', 'sub'),
        ('quaternium-15', 'sub')
    ],
    'fragrance': [
        ('fragrance', 'sub'), ('parfum', 'sub'), ('perfume', 'sub')
    ],
    'oxybenzone': [
        ('oxybenzone', 'sub'), ('benzophenone-3', 'sub')
    ],
    'sulfates': [
        ('sodium lauryl sulfate', 'sub'),
        ('sodium laureth sulfate', 'sub'),
        ('sls', 'word'),
        ('sles', 'word')
    ],
    'heavy metals/pigments': [
        ('lead', 'word'), ('mercury', 'word'),
        ('arsenic', 'word'), ('cadmium', 'word'),
        ('chromium', 'word'), ('iron oxides', 'sub'),
        ('titanium dioxide', 'sub')
    ]
}

def matches_term(ingredient: str, term: str, mode: str) -> bool:
    """Substring or whole-word match on a lowercased ingredient name."""
    ing_lower = ingredient.lower()
    if mode == 'word':
        return re.search(rf'\b{re.escape(term)}\b', ing_lower) is not None
    return term in ing_lower

risk_detection_records = []

print(f"  {'Category':<30} {'Variants':>10} {'Mentions':>12}")
print(f"  {'--------':<30} {'--------':>10} {'--------':>12}")

for category, search_terms in key_ingredients.items():
    matches = []
    for ing in unique_ingredients:
        for term, mode in search_terms:
            if matches_term(ing, term, mode):
                matches.append(ing)
                break

    matches        = sorted(set(matches))
    total_mentions = sum(freq[m] for m in matches)

    print(f"  {category:<30} {len(matches):>10,} "
          f"{total_mentions:>12,}")
    if matches:
        print(f"    Examples: {matches[:4]}")

    for m in matches:
        risk_detection_records.append({
            'risk_category':      category,
            'matched_ingredient': m,
            'frequency':          freq[m]
        })

risk_detection_df = pd.DataFrame(risk_detection_records)
risk_detection_df.to_csv(
    os.path.join(QUALITY_PATH,
                 'key_risk_ingredient_detection.csv'),
    index=False
)



STEP 5: Key Risk Ingredient Detection

  Category                         Variants     Mentions
  --------                         --------     --------
  parabens                               13          332
    Examples: ['100% cotton wick. paraben', '100% cotton wick. paraben-', 'butylparaben', 'ethylparaben']
  phthalates                             11           99
    Examples: ['6-naphthalate', 'c264217/1 polyethylene terephthalate', 'made from high-quality pvc (vinyl). free from harmful chemicals such as phthalates', 'phthalate']
  formaldehyde releasers                  6           44
    Examples: ['diazolidinyl urea', 'dmdm hydantoin', 'formaldehyde cyclododecyl ethyl acetal', 'formaldehyde cyclododecyl ethyl acetal (woody note/safe synthetic)']
  fragrance                             112        4,360
    Examples: ['(fragrance)', '*fragrance (parfum)', '2 hexanediol parfum (fragrance)', '\\rparfum (fragrance)']
  oxybenzone                              6           58
    E

In [8]:
# ----------------------------------------------------------
# STEP 6: INGREDIENT LENGTH DISTRIBUTION
# ----------------------------------------------------------

print()
print("STEP 6: Ingredient Length Distribution")
print()

length_records = pd.DataFrame({
    'ingredient': unique_ingredients,
    'frequency':  [freq[ing] for ing in unique_ingredients],
    'length':     [len(ing) for ing in unique_ingredients]
})

print(f"  {'Statistic':<25} {'Value':>12}")
print(f"  {'----------':<25} {'-----':>12}")
print(f"  {'Minimum':<25} "
      f"{length_records['length'].min():>12}")
print(f"  {'25th percentile':<25} "
      f"{length_records['length'].quantile(0.25):>12.0f}")
print(f"  {'Median':<25} "
      f"{length_records['length'].median():>12.0f}")
print(f"  {'75th percentile':<25} "
      f"{length_records['length'].quantile(0.75):>12.0f}")
print(f"  {'Maximum':<25} "
      f"{length_records['length'].max():>12}")
print(f"  {'Mean':<25} "
      f"{length_records['length'].mean():>12.1f}")
print(f"  {'Std Dev':<25} "
      f"{length_records['length'].std():>12.1f}")


long_ingredients = length_records[
    length_records['length'] > 80
].sort_values(['frequency', 'length'],
              ascending=[False, False])

print()
print(f"  {'Very long ingredients >80 chars':<40} "
      f"{len(long_ingredients):>8,}")

long_ingredients.to_csv(
    os.path.join(QUALITY_PATH, 'very_long_ingredients.csv'),
    index=False
)



STEP 6: Ingredient Length Distribution

  Statistic                        Value
  ----------                       -----
  Minimum                              3
  25th percentile                     17
  Median                              27
  75th percentile                     37
  Maximum                           1033
  Mean                              30.0
  Std Dev                           29.3

  Very long ingredients >80 chars               149


In [9]:
# ----------------------------------------------------------
# STEP 7: COVERAGE ANALYSIS
# ----------------------------------------------------------

print()
print("STEP 7: Ingredient Coverage Analysis")
print()
print("  This table determines how many ingredients")
print("  must be classified to achieve target coverage.")
print()

total_mentions = len(all_ingredients)

print(f"  {'Top N':<20} {'Unique %':>12} {'Coverage %':>12}")
print(f"  {'-----':<20} {'--------':>12} {'----------':>12}")

coverage_records = []

for n in [100, 200, 300, 400, 500]:
    top_n_list       = freq.most_common(n)
    mentions_covered = sum(c for _, c in top_n_list)
    unique_pct       = n / len(unique_ingredients) * 100
    coverage_pct     = mentions_covered / total_mentions * 100

    coverage_records.append({
        'top_n':               n,
        'unique_percentage':   round(unique_pct, 2),
        'coverage_percentage': round(coverage_pct, 2)
    })

    print(f"  {n:<20,} {unique_pct:>11.1f}% "
          f"{coverage_pct:>11.1f}%")

coverage_df = pd.DataFrame(coverage_records)
coverage_df.to_csv(
    os.path.join(QUALITY_PATH,
                 'risk_dictionary_coverage_analysis.csv'),
    index=False
)

print()



STEP 7: Ingredient Coverage Analysis

  This table determines how many ingredients
  must be classified to achieve target coverage.

  Top N                    Unique %   Coverage %
  -----                    --------   ----------
  100                          0.7%        43.6%
  200                          1.4%        56.0%
  300                          2.2%        62.7%
  400                          2.9%        67.4%
  500                          3.6%        70.9%



In [10]:
# ----------------------------------------------------------
# STEP 8: FINAL QUALITY CHECK SUMMARY
# ----------------------------------------------------------

print()
print("STEP 8: Final Quality Check Summary")
print()

rows = [
    ('Products reviewed',
     len(products)),
    ('Upstream ingredient mentions',
     upstream_mentions),
    ('Reconstructed ingredient mentions',
     reconstructed_mentions),
    ('Reconstruction difference',
     f"{mention_difference:+,}"),
    ('Products with count differences',
     mismatch_count),
    ('Count-check status',
     count_check_status),
    ('Total unique ingredients',
     len(unique_ingredients)),
    ('Low ingredient count products',
     len(low_count_products)),
    ('Potential parsing artifacts',
     len(artifacts_df)),
    ('Artifact rate',
     f"{artifact_rate:.1f}%"),
    ('Non-ingredient phrase candidates',
     len(phrase_flags_df)),
    ('Very long ingredients >80 chars',
     len(long_ingredients)),
]

print(f"  {'Check':<40} {'Value':>12}")
print(f"  {'----------':<40} {'-----':>12}")
for label, value in rows:
    if isinstance(value, int):
        print(f"  {label:<40} {value:>12,}")
    else:
        print(f"  {label:<40} {value:>12}")

summary_df = pd.DataFrame(
    [(r[0], str(r[1])) for r in rows],
    columns=['Check', 'Value']
)
summary_df.to_csv(
    os.path.join(QUALITY_PATH,
                 'ingredient_quality_summary.csv'),
    index=False
)

print()
print(f"  Files saved to: {QUALITY_PATH}")
print()

files = [
    'low_ingredient_count_products.csv',
    'potential_parsing_artifacts.csv',
    'non_ingredient_phrase_candidates.csv',
    'ingredient_variant_groups.csv',
    'key_risk_ingredient_detection.csv',
    'very_long_ingredients.csv',
    'risk_dictionary_coverage_analysis.csv',
    'ingredient_quality_summary.csv'
]

for f in files:
    print(f"  - {f}")

print()



STEP 8: Final Quality Check Summary

  Check                                           Value
  ----------                                      -----
  Products reviewed                               7,544
  Upstream ingredient mentions                  261,559
  Reconstructed ingredient mentions             261,561
  Reconstruction difference                          +2
  Products with count differences                     1
  Count-check status                       Known two-entry reconstruction difference
  Total unique ingredients                       13,899
  Low ingredient count products                     111
  Potential parsing artifacts                       168
  Artifact rate                                    1.2%
  Non-ingredient phrase candidates                  100
  Very long ingredients >80 chars                   149

  Files saved to: ../data/quality_check

  - low_ingredient_count_products.csv
  - potential_parsing_artifacts.csv
  - non_ingredient_phrase_candida